# cuTile Python: Activation Functions

The previous modules introduced 2D tiles through matrix addition and transpose. This module uses the same tile structure for activation functions: ReLU and clamp.

Standard Python operators like `+` and `*` apply elementwise across a tile automatically. For operations that do not have a Python operator equivalent, cuTile provides named functions like `ct.maximum` and `ct.minimum`.

## Installing cuTile Python

cuTile Python requires:

    - NVIDIA Kernel Driver R580 or later.
    - CUDA Toolkit 13.1 or later.

You can install cuTile Python via the PIP package `cuda-tile`.

In [ ]:
import os

if not os.getenv("BREV_ENV_ID") and not os.path.exists(os.path.expanduser("~/.accelerated-computing-hub-installed")): # If not running in brev
  print("Installing PIP packages.")
  !pip uninstall "cuda-python" --yes > /dev/null
  !pip install "cuda-tile" "cupy-cuda13x" > /dev/null 2>&1
  open(os.path.expanduser("~/.accelerated-computing-hub-installed"), "a").close()

In [ ]:
import cuda.tile as ct
import cupy as cp

## Example: ReLU

ReLU replaces every negative number with zero and leaves positive numbers unchanged:

$$y = \max(x,\; 0)$$

It is one of the most widely used activation functions in neural networks.

`ct.maximum(tile, value)` applies a max elementwise across every element in the tile.
Passing `0` sets all negatives to zero in a single call.

In [ ]:
@ct.kernel
def relu(X: ct.Array, Y: ct.Array, tm: ct.Constant[int], tn: ct.Constant[int]):
  row = ct.bid(0)
  col = ct.bid(1)

  x_tile = ct.load(X, index=(row, col), shape=(tm, tn))

  ct.store(Y, index=(row, col), tile=ct.maximum(x_tile, 0))

In [ ]:
X = cp.random.uniform(-5, 5, (1024, 1024), dtype=cp.float32)
Y = cp.zeros_like(X)

tm, tn = 64, 64
grid = (ct.cdiv(X.shape[0], tm), ct.cdiv(X.shape[1], tn), 1)
print(f"Grid: {grid[0]} × {grid[1]} = {grid[0] * grid[1]:,} blocks")

ct.launch(cp.cuda.get_current_stream(), grid, relu, (X, Y, tm, tn))

cp.testing.assert_array_almost_equal(Y, cp.maximum(X, 0))
print("ReLU OK")

Notice how similar this is to the matrix add kernel from module 02. The load-transform-store structure is identical. The only difference is the operation: `ct.maximum(x_tile, 0)` instead of `a_tile + b_tile`.

## Exercise: Clamp

Clamp limits every value to a range `[lo, hi]`:

$$y = \min(\max(x,\; lo),\; hi)$$

It is used in gradient clipping and quantization.
`ct.minimum(tile, value)` works the same way as `ct.maximum` but applies a min instead of a max.

Write a kernel that clamps a matrix to `[-1.0, 1.0]`.

Give the exercise a try. A reference solution follows after.

In [ ]:
@ct.kernel
def clamp(X: ct.Array, Y: ct.Array, tm: ct.Constant[int], tn: ct.Constant[int]):
  # TODO: your code here.
  # Hint: nest ct.minimum and ct.maximum.
  pass


# X2 = cp.random.uniform(-10, 10, (1024, 1024), dtype=cp.float32)
# Y2 = cp.zeros_like(X2)
# tm, tn = 64, 64
# grid = (ct.cdiv(X2.shape[0], tm), ct.cdiv(X2.shape[1], tn), 1)
# ct.launch(cp.cuda.get_current_stream(), grid, clamp, (X2, Y2, tm, tn))
# cp.testing.assert_array_almost_equal(Y2, cp.clip(X2, -1.0, 1.0))
# print("Exercise OK")

### Solution

Run the cell below to check your work.

In [ ]:
@ct.kernel
def _clamp(X: ct.Array, Y: ct.Array, tm: ct.Constant[int], tn: ct.Constant[int]):
  row = ct.bid(0)
  col = ct.bid(1)

  x_tile = ct.load(X, index=(row, col), shape=(tm, tn))

  ct.store(Y, index=(row, col), tile=ct.minimum(ct.maximum(x_tile, -1.0), 1.0))


X2 = cp.random.uniform(-10, 10, (1024, 1024), dtype=cp.float32)
Y2 = cp.zeros_like(X2)

tm, tn = 64, 64
grid = (ct.cdiv(X2.shape[0], tm), ct.cdiv(X2.shape[1], tn), 1)
ct.launch(cp.cuda.get_current_stream(), grid, _clamp, (X2, Y2, tm, tn))

cp.testing.assert_array_almost_equal(Y2, cp.clip(X2, -1.0, 1.0))
print("Exercise OK")

ReLU and clamp use named cuTile operations: `ct.maximum` and `ct.minimum`. For the full list, see the [operations reference](https://docs.nvidia.com/cuda/cutile-python/operations.html).